# ConvNeXt Semantic Segmentation — FCN / UPerNet (PyTorch)

**Backbone:** ConvNeXt-Tiny (pretrained on ImageNet)  
**Segmentation Head:** UPerNet-style decoder (PPM + FPN fusion)  
**Dataset:** PASCAL VOC 2012 Segmentation (21 classes including background)

**Classes:** background, aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow, dining table, dog, horse, motorbike, person, potted plant, sheep, sofa, train, tv/monitor

**Architecture:**
```
ConvNeXt-Tiny (backbone)
    ↓ (multi-scale features: C1, C2, C3, C4)
PPM (Pyramid Pooling Module) on C4
    ↓
FPN-style lateral fusion (C1–C4)
    ↓
Concat + Conv → Pixel-wise Classification (H × W × 21)
```

> GPU accelerator ဖွင့်ပြီး run ပါ: `Settings → Accelerator → GPU`

In [ ]:
# --- 1. Install Dependencies ---
# !pip install -q torch torchvision matplotlib tqdm scikit-learn

In [ ]:
# --- 2. Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
import random
import time
from PIL import Image
from tqdm import tqdm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("✅ All imports successful.")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- 3. Device & Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hyperparameters
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-5
NUM_WORKERS = 2
FREEZE_EPOCHS = 5
NUM_CLASSES = 21  # VOC: 20 classes + background

# VOC segmentation class names
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog',
    'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa',
    'train', 'tvmonitor'
]

# VOC color palette for visualization (21 classes)
VOC_COLORMAP = np.array([
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
    [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0],
    [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128],
    [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
    [0, 64, 128]
], dtype=np.uint8)

print(f"Classes ({NUM_CLASSES}): {VOC_CLASSES}")

## 📦 Dataset — PASCAL VOC 2012 Segmentation

VOC segmentation dataset ကို load လုပ်ပါမယ်။  
- Input: RGB image  
- Target: Pixel-wise class label mask (0–20, 255=ignore)

In [ ]:
# --- 4. VOC Segmentation Dataset ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class VOCSegmentationDataset(Dataset):
    """PASCAL VOC Segmentation dataset wrapper."""
    def __init__(self, root, year='2012', image_set='train', img_size=512):
        self.voc = torchvision.datasets.VOCSegmentation(
            root=root, year=year, image_set=image_set, download=True
        )
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        img, mask = self.voc[idx]

        # Transform image
        img = self.img_transform(img)

        # Resize mask with NEAREST (no interpolation for labels!)
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)
        mask = torch.from_numpy(np.array(mask)).long()

        # VOC uses 255 as "ignore" boundary — map to 255 (ignore_index)
        mask[mask == 255] = 255

        return img, mask

# Download & Load
print("⏳ Downloading PASCAL VOC 2012 Segmentation...")
train_dataset = VOCSegmentationDataset('./data', year='2012', image_set='train', img_size=IMG_SIZE)
val_dataset = VOCSegmentationDataset('./data', year='2012', image_set='val', img_size=IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print(f"   Batches → Train: {len(train_loader)} | Val: {len(val_loader)}")

In [ ]:
# --- 5. Visualize Samples (Image + Segmentation Mask) ---
def unnormalize(img, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    img = img.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(img, 0, 1)

def colorize_mask(mask, colormap=VOC_COLORMAP):
    """Convert class-index mask to RGB color mask."""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(len(colormap)):
        color_mask[mask == cls_id] = colormap[cls_id]
    return color_mask

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i in range(3):
    idx = random.randint(0, len(train_dataset) - 1)
    img, mask = train_dataset[idx]
    img_display = unnormalize(img).permute(1, 2, 0).numpy()
    mask_np = mask.numpy()
    color_mask = colorize_mask(mask_np)

    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title("Input Image", fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_np, cmap='tab20', vmin=0, vmax=20)
    axes[i, 1].set_title("Class Mask (Raw)", fontsize=10)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(color_mask)
    axes[i, 2].set_title("Color Segmentation Mask", fontsize=10)
    axes[i, 2].axis('off')

    # Overlay
    overlay = 0.6 * img_display + 0.4 * (color_mask / 255.0)
    axes[i, 3].imshow(np.clip(overlay, 0, 1))
    axes[i, 3].set_title("Overlay", fontsize=10)
    axes[i, 3].axis('off')

plt.suptitle("VOC 2012 Segmentation Samples — Image | Mask | Color | Overlay",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏗️ ConvNeXt + UPerNet Segmentation Model

UPerNet (Unified Perceptual Parsing Network) decoder ကို ConvNeXt backbone နဲ့ ချိတ်ဆက်ပါမယ်။

```
ConvNeXt-Tiny Backbone → 4 stages (C1=96, C2=192, C3=384, C4=768)
    ↓
PPM (Pyramid Pooling Module) on C4
    ↓
FPN lateral connections (C1–C4 + PPM)
    ↓
Upsample + Concat → Conv → Pixel Classification (H × W × 21)
```

In [ ]:
# --- 6. ConvNeXt Backbone (Multi-Scale Feature Extractor) ---
class ConvNeXtBackbone(nn.Module):
    """Extract multi-scale features from ConvNeXt-Tiny."""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
        convnext = models.convnext_tiny(weights=weights)
        features = convnext.features

        self.stage1 = nn.Sequential(features[0], features[1])  # 96ch, H/4
        self.stage2 = nn.Sequential(features[2], features[3])  # 192ch, H/8
        self.stage3 = nn.Sequential(features[4], features[5])  # 384ch, H/16
        self.stage4 = nn.Sequential(features[6], features[7])  # 768ch, H/32

        self.channels = [96, 192, 384, 768]

    def forward(self, x):
        c1 = self.stage1(x)
        c2 = self.stage2(c1)
        c3 = self.stage3(c2)
        c4 = self.stage4(c3)
        return [c1, c2, c3, c4]


# --- 7. PPM (Pyramid Pooling Module) ---
class PPM(nn.Module):
    """Pyramid Pooling Module from PSPNet."""
    def __init__(self, in_channels, pool_sizes=[1, 2, 3, 6]):
        super().__init__()
        out_channels = in_channels // len(pool_sizes)
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(ps),
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ) for ps in pool_sizes
        ])
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_channels + out_channels * len(pool_sizes), in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        h, w = x.shape[2:]
        pyramids = [x]
        for stage in self.stages:
            pyramids.append(F.interpolate(stage(x), size=(h, w), mode='bilinear', align_corners=False))
        return self.bottleneck(torch.cat(pyramids, dim=1))


# --- 8. UPerNet Decoder ---
class UPerNetDecoder(nn.Module):
    """UPerNet-style decoder with FPN fusion."""
    def __init__(self, in_channels_list, fpn_channels=256, num_classes=21):
        super().__init__()

        # PPM on the deepest features
        self.ppm = PPM(in_channels_list[-1])

        # Lateral convolutions (1x1 to match FPN channels)
        self.lateral_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(ch, fpn_channels, 1, bias=False),
                nn.BatchNorm2d(fpn_channels),
                nn.ReLU(inplace=True)
            ) for ch in in_channels_list
        ])

        # FPN output convolutions (3x3 after addition)
        self.fpn_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(fpn_channels),
                nn.ReLU(inplace=True)
            ) for _ in in_channels_list
        ])

        # Final fusion
        self.fpn_bottleneck = nn.Sequential(
            nn.Conv2d(fpn_channels * len(in_channels_list), fpn_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )

        # Classification head
        self.cls_head = nn.Conv2d(fpn_channels, num_classes, 1)

    def forward(self, features):
        # Apply PPM on deepest feature
        features[-1] = self.ppm(features[-1])

        # Lateral connections
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        # Top-down FPN fusion
        for i in range(len(laterals) - 1, 0, -1):
            h, w = laterals[i - 1].shape[2:]
            laterals[i - 1] = laterals[i - 1] + F.interpolate(
                laterals[i], size=(h, w), mode='bilinear', align_corners=False
            )

        # FPN output convs
        fpn_outs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]

        # Upsample all to largest size and concatenate
        target_size = fpn_outs[0].shape[2:]
        fpn_outs_resized = [
            F.interpolate(out, size=target_size, mode='bilinear', align_corners=False)
            for out in fpn_outs
        ]
        fused = torch.cat(fpn_outs_resized, dim=1)
        fused = self.fpn_bottleneck(fused)

        # Classification
        out = self.cls_head(fused)
        return out


# --- 9. Full Segmentation Model ---
class ConvNeXtUPerNet(nn.Module):
    """ConvNeXt + UPerNet for Semantic Segmentation."""
    def __init__(self, num_classes=21, pretrained_backbone=True):
        super().__init__()
        self.backbone = ConvNeXtBackbone(pretrained=pretrained_backbone)
        self.decoder = UPerNetDecoder(
            in_channels_list=self.backbone.channels,
            fpn_channels=256,
            num_classes=num_classes
        )

    def forward(self, x):
        input_size = x.shape[2:]
        features = self.backbone(x)
        out = self.decoder(features)
        # Upsample to input size
        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)
        return out

model = ConvNeXtUPerNet(num_classes=NUM_CLASSES, pretrained_backbone=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:>12,}")
print(f"Trainable Parameters: {trainable_params:>12,}")
print(f"✅ ConvNeXt + UPerNet built successfully.")

In [ ]:
# --- 10. Loss, Optimizer & Metrics ---
# Freeze backbone initially
for param in model.backbone.parameters():
    param.requires_grad = False

trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 — Trainable params (decoder only): {trainable_p:,}")

# Loss (ignore index 255 = VOC boundary pixels)
criterion = nn.CrossEntropyLoss(ignore_index=255)

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)


def compute_miou(pred, target, num_classes=NUM_CLASSES, ignore_index=255):
    """Compute mean IoU."""
    pred = pred.argmax(dim=1).cpu().numpy()
    target = target.cpu().numpy()
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        # Ignore boundary pixels
        valid = (target != ignore_index)
        pred_cls = pred_cls & valid
        target_cls = target_cls & valid
        intersection = (pred_cls & target_cls).sum()
        union = (pred_cls | target_cls).sum()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0


def pixel_accuracy(pred, target, ignore_index=255):
    """Compute pixel accuracy."""
    pred = pred.argmax(dim=1)
    valid = (target != ignore_index)
    correct = ((pred == target) & valid).sum().item()
    total = valid.sum().item()
    return correct / total if total > 0 else 0.0

print("✅ Loss, Optimizer, Metrics configured.")

## 🚀 Training Loop

**2-Phase Training:**
1. Phase 1: Backbone frozen → decoder (UPerNet) only train
2. Phase 2: Unfreeze backbone → full fine-tune with low LR

**Metrics:**
- Training/Validation Loss
- Pixel Accuracy
- Mean IoU (mIoU)

In [ ]:
# --- 11. Training Loop ---
train_losses, val_losses = [], []
train_mious, val_mious = [], []
train_accs, val_accs = [], []
best_miou = 0.0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # === Phase 2: Unfreeze backbone ===
    if epoch == FREEZE_EPOCHS:
        print(f"\n{'='*60}")
        print("🔓 Phase 2: Unfreezing ConvNeXt backbone for fine-tuning!")
        print(f"{'='*60}")
        for param in model.backbone.parameters():
            param.requires_grad = True

        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': FINE_TUNE_LR},
            {'params': model.decoder.parameters(), 'lr': LEARNING_RATE * 0.5},
        ], weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Trainable parameters: {trainable:,}\n")

    phase = "Phase 1 (Frozen)" if epoch < FREEZE_EPOCHS else "Phase 2 (Fine-tune)"

    # === Train ===
    model.train()
    epoch_loss, epoch_miou, epoch_acc = 0.0, 0.0, 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)       # (B, 21, H, W)
        loss = criterion(outputs, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_miou += compute_miou(outputs.detach(), masks)
        epoch_acc += pixel_accuracy(outputs.detach(), masks)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = epoch_loss / len(train_loader)
    train_miou = epoch_miou / len(train_loader)
    train_acc = epoch_acc / len(train_loader)
    train_losses.append(train_loss)
    train_mious.append(train_miou)
    train_accs.append(train_acc)

    # === Validate ===
    model.eval()
    v_loss, v_miou, v_acc = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            v_loss += criterion(outputs, masks).item()
            v_miou += compute_miou(outputs, masks)
            v_acc += pixel_accuracy(outputs, masks)

    val_loss = v_loss / len(val_loader)
    val_miou = v_miou / len(val_loader)
    val_acc = v_acc / len(val_loader)
    val_losses.append(val_loss)
    val_mious.append(val_miou)
    val_accs.append(val_acc)

    scheduler.step(val_miou)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"[{phase}] Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | mIoU: {train_miou:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.2e}")

    if val_miou > best_miou:
        best_miou = val_miou
        torch.save(model.state_dict(), "best_convnext_upernet_voc.pth")
        print(f"  → 💾 Best model saved! (mIoU: {val_miou:.4f})")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ Training Complete! Time: {elapsed/60:.1f} min")
print(f"   Best Validation mIoU: {best_miou:.4f}")
print(f"{'='*60}")

In [ ]:
# --- 12. Plot Training History ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss
axes[0].plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(range(1, NUM_EPOCHS+1), val_losses, 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per Epoch', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# mIoU
axes[1].plot(range(1, NUM_EPOCHS+1), train_mious, 'b-o', label='Train mIoU', markersize=4)
axes[1].plot(range(1, NUM_EPOCHS+1), val_mious, 'r-o', label='Val mIoU', markersize=4)
axes[1].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Mean IoU per Epoch', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Pixel Accuracy
axes[2].plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train Acc', markersize=4)
axes[2].plot(range(1, NUM_EPOCHS+1), val_accs, 'r-o', label='Val Acc', markersize=4)
axes[2].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Pixel Accuracy')
axes[2].set_title('Pixel Accuracy per Epoch', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle("ConvNeXt + UPerNet — Training History", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("convnext_segmentation_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 🧪 Evaluation & Visualization

Best model ကို load ပြီး validation set ပေါ်မှာ segmentation results ကို visualize လုပ်ပါမယ်။

In [ ]:
# --- 13. Load Best Model & Evaluate ---
model.load_state_dict(torch.load("best_convnext_upernet_voc.pth",
                                  map_location=device, weights_only=True))
model.eval()

# Per-class IoU on validation set
per_class_intersection = np.zeros(NUM_CLASSES)
per_class_union = np.zeros(NUM_CLASSES)
total_correct = 0
total_pixels = 0

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="Evaluating"):
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        targets = masks.cpu().numpy()

        valid = (targets != 255)
        total_correct += ((preds == targets) & valid).sum()
        total_pixels += valid.sum()

        for cls in range(NUM_CLASSES):
            pred_cls = (preds == cls) & valid
            target_cls = (targets == cls) & valid
            per_class_intersection[cls] += (pred_cls & target_cls).sum()
            per_class_union[cls] += (pred_cls | target_cls).sum()

# Compute per-class IoU
per_class_iou = np.zeros(NUM_CLASSES)
for cls in range(NUM_CLASSES):
    if per_class_union[cls] > 0:
        per_class_iou[cls] = per_class_intersection[cls] / per_class_union[cls]

overall_acc = total_correct / total_pixels
mean_iou = per_class_iou[per_class_union > 0].mean()

print(f"\n{'='*55}")
print(f"{'Class':<20} {'IoU':>10}")
print(f"{'='*55}")
for i, cls in enumerate(VOC_CLASSES):
    if per_class_union[i] > 0:
        print(f"{cls:<20} {per_class_iou[i]:>10.4f}")
print(f"{'='*55}")
print(f"{'Pixel Accuracy':<20} {overall_acc:>10.4f}")
print(f"{'Mean IoU':<20} {mean_iou:>10.4f}")
print(f"{'='*55}")

In [ ]:
# --- 14. Per-Class IoU Bar Chart ---
valid_classes = per_class_union > 0
valid_names = [VOC_CLASSES[i] for i in range(NUM_CLASSES) if valid_classes[i]]
valid_ious = per_class_iou[valid_classes]

sorted_idx = np.argsort(valid_ious)[::-1]
sorted_names = [valid_names[i] for i in sorted_idx]
sorted_ious = valid_ious[sorted_idx]

plt.figure(figsize=(12, 6))
colors = plt.cm.RdYlGn(sorted_ious[::-1])  # Green=high, Red=low
bars = plt.barh(sorted_names[::-1], sorted_ious[::-1], color=colors, edgecolor='white')
plt.xlabel('IoU', fontsize=12)
plt.title(f'Per-Class IoU — ConvNeXt + UPerNet (mIoU: {mean_iou:.4f})',
          fontsize=13, fontweight='bold')
plt.xlim(0, 1)
for bar, val in zip(bars, sorted_ious[::-1]):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig("convnext_segmentation_iou.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 15. Visualize Segmentation Results ---
def show_segmentation_results(model, dataset, num_samples=4):
    """Show input, ground truth mask, prediction, and overlay."""
    model.eval()
    fig, axes = plt.subplots(num_samples, 5, figsize=(22, 5 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for row, idx in enumerate(indices):
            img, gt_mask = dataset[idx]
            output = model(img.unsqueeze(0).to(device))
            pred_mask = output.argmax(dim=1).squeeze().cpu().numpy()

            img_display = unnormalize(img).permute(1, 2, 0).numpy()
            gt_mask_np = gt_mask.numpy()
            gt_color = colorize_mask(gt_mask_np)
            pred_color = colorize_mask(pred_mask)

            # (A) Input
            axes[row, 0].imshow(img_display)
            axes[row, 0].set_title("Input Image", fontsize=10)
            axes[row, 0].axis('off')

            # (B) Ground Truth Mask
            axes[row, 1].imshow(gt_color)
            axes[row, 1].set_title("Ground Truth", fontsize=10)
            axes[row, 1].axis('off')

            # (C) Predicted Mask
            axes[row, 2].imshow(pred_color)
            axes[row, 2].set_title("Prediction", fontsize=10)
            axes[row, 2].axis('off')

            # (D) Overlay prediction on image
            overlay = 0.5 * img_display + 0.5 * (pred_color / 255.0)
            axes[row, 3].imshow(np.clip(overlay, 0, 1))
            axes[row, 3].set_title("Pred Overlay", fontsize=10)
            axes[row, 3].axis('off')

            # (E) Error map (red = wrong, green = correct)
            valid = gt_mask_np != 255
            error = np.zeros((*gt_mask_np.shape, 3), dtype=np.float32)
            correct = (pred_mask == gt_mask_np) & valid
            wrong = (pred_mask != gt_mask_np) & valid
            error[correct] = [0, 0.8, 0]   # green
            error[wrong] = [0.9, 0, 0]     # red
            error[~valid] = [0.3, 0.3, 0.3] # gray (ignore)
            axes[row, 4].imshow(error)
            acc = correct.sum() / valid.sum() if valid.sum() > 0 else 0
            axes[row, 4].set_title(f"Error Map (Acc: {acc:.2%})", fontsize=10)
            axes[row, 4].axis('off')

    plt.suptitle("ConvNeXt + UPerNet — Segmentation Results\nInput | GT | Pred | Overlay | Error",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("convnext_segmentation_results.png", dpi=150, bbox_inches='tight')
    plt.show()

show_segmentation_results(model, val_dataset, num_samples=4)

In [ ]:
# --- 16. Single Image Segmentation Inference ---
def segment_image(model, image_path):
    """Run segmentation on a single image and display results."""
    model.eval()
    img_pil = Image.open(image_path).convert('RGB')
    original_size = img_pil.size  # (W, H)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    img_tensor = transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        pred = output.argmax(dim=1).squeeze().cpu().numpy()

    pred_color = colorize_mask(pred)
    img_display = unnormalize(img_tensor.squeeze().cpu()).permute(1, 2, 0).numpy()

    # Find detected classes
    unique_classes = np.unique(pred)
    detected = [VOC_CLASSES[c] for c in unique_classes if c != 0]  # skip background

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_pil)
    axes[0].set_title("Original Image", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(pred_color)
    axes[1].set_title("Segmentation Mask", fontsize=12, fontweight='bold')
    axes[1].axis('off')

    overlay = 0.5 * img_display + 0.5 * (pred_color / 255.0)
    axes[2].imshow(np.clip(overlay, 0, 1))
    axes[2].set_title(f"Overlay — Detected: {', '.join(detected) if detected else 'background only'}",
                      fontsize=11, fontweight='bold')
    axes[2].axis('off')

    plt.suptitle("ConvNeXt + UPerNet — Single Image Segmentation", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Detected classes: {detected}")
    return pred, detected

# Usage:
# segment_image(model, "path/to/image.jpg")
print("✅ segment_image() ready. Usage:")
print('   segment_image(model, "path/to/image.jpg")')

In [ ]:
# --- 17. Color Legend ---
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
for i, (cls, color) in enumerate(zip(VOC_CLASSES, VOC_COLORMAP)):
    ax.barh(i, 1, color=color/255.0, edgecolor='white', height=0.8)
    ax.text(0.5, i, cls, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white' if np.mean(color) < 128 else 'black')
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(0, 1)
ax.set_title("VOC Segmentation Color Legend", fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()